# Packages

In [ ]:
import pandas as pd
import os, sys

In [ ]:
#Versions Used For This Data Managment Script
print(f"panads: {pd.__version__}") #2.2.2
print(f"Platform System: {os.name}") #posix
print(f"Python: {sys.version}") #3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:48) [Clang 13.0.0 (clang-1300.0.29.30)]

# Flags

In [ ]:
preprocess_raw_data = False

debug = True

output_data = True

output_ver = 9

# Data Selection and Output

In [ ]:
data_dir = "../raw-data/experiment-data/" #where all data for TMS study is stored

treatment_map = "../raw-data/participant_tracking.csv" #the csv that contains what treatment was done per participant

program_grades = "../raw-data/gradebook.csv" #the csv contains the outcome to programming responses

demographics = "../raw-data/demographics.csv" 

pain_reports = "../raw-data/pain_reports.csv"

output_dir = f"../preprocessed-data/full-data-version-{output_ver}"


Create output directory if it doesn't already exist. (This is where all your preprocessed data will go - nback, FITB, LR)

In [ ]:
#Create output diretory
if output_data == True:
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Output Directory '{output_dir}' created.")
    elif os.path.exists(output_dir):
        print(f"Output Directory '{output_dir}' already exists.")


In [ ]:
if debug:
    #Check data from participant exists
    data_test = pd.read_csv(data_dir+"A23A-session-2/one_back_section.csv", sep="@")
    display(data_test.head(3))

# Data Preproccessing

### Step 1: Correcting N-back Data 

Some data sheets wrongly caluclated correctness for participants. We correct this here.

NOTE -  IF "...corrected.csv" FILES ALREADY EXISTS, YOU DO NOT NEED TO RUN THIS.

In [ ]:
#------------------------------------------------------
#---- Function that fixed the n-back miscaluclation ---
#------------------------------------------------------

def fix_expected_key(file_path, n_back):
    # Load the CSV
    df = pd.read_csv(file_path, sep="@")

    #Output file
    output_file = file_path.replace('.csv', '_corrected.csv')
    if os.path.exists(output_file): 
        print(f"{output_file} already exists - Do you mean to delete it?")
        return 0
    
    # Create corrected columns
    corrected_is_nback = []
    corrected_expected_key = []
    corrected_correct_answer = []

    # Iterate through each row
    for idx in range(len(df)):
        if idx >= n_back:  # Ensure there's enough prior data
            # Check if the current letter is n-back
            is_nback = df['shown_letter'][idx] == df['shown_letter'][idx - n_back]
            corrected_is_nback.append(is_nback)

            # Determine the expected key
            expected_key = "m" if is_nback else "z"
            corrected_expected_key.append(expected_key)

        else:
            # Not enough data to calculate n-back
            is_nback = False
            expected_key = "z"
            corrected_is_nback.append(is_nback)
            corrected_expected_key.append(expected_key)
        
         
        # Check participant's correctness
        chosen_key = df['chosen_key'][idx]
        correct_answer = chosen_key == expected_key
        corrected_correct_answer.append(correct_answer)

    # Add corrected columns back to the DataFrame
    df['nback_type'] = str(n_back)+"-back"
    df['is_nback'] = corrected_is_nback
    df['expected_key'] = corrected_expected_key
    df['correct_answer'] = corrected_correct_answer


    # Save the fixed file (optional)
    df.to_csv(output_file, sep="@", index=False)

    return 1



In [ ]:
#(ALREADY DONE FOR EVERYONE SO NOT NEEDED FOR REPLICATION)
#------------------------------------------------------
#---- Fixes all nback data for participants such that we accuratly assess n-back 
#------------------------------------------------------

if preprocess_raw_data:

    participant_folders = os.listdir(data_dir)

    for session_folders in participant_folders:
        data_files = os.listdir(data_dir+session_folders)
        for file in data_files:
            if "back_section.csv" in file:
                nback = file.split("_")[0]
                if nback == "three": nback = 3
                elif nback == "two": nback = 2
                elif nback == "one": nback = 1
                #else: print("WARNING: nback CSV is none of the three!!!")
                if debug: print(data_dir+session_folders+"/"+file)
                fix_expected_key(data_dir+session_folders+"/"+file, nback)

    print(f"Data in {data_dir} was preprocessed such that n_back csvs are converted to 'corrected.csv'.")

### Step 2: Merge N-back Data

In [ ]:
dataframes = []
participant_folders = os.listdir(data_dir)
for session_folder in participant_folders:
        data_files = os.listdir(data_dir+session_folder)
        for file in data_files:
            if "back_section_corrected" in file:
                  df = pd.read_csv(data_dir+session_folder+"/"+file, sep="@")
                  dataframes.append(df)

# Concatenate all DataFrames into one large DataFrame
nback_full_data = pd.concat(dataframes, ignore_index=True)



In [ ]:
#Manually check n-back data frame
if debug:
    display(nback_full_data)

### Step 3: Process FITB data

In [ ]:
dataframes = []
participant_folders = os.listdir(data_dir)
for session_folder in participant_folders:
        data_files = os.listdir(data_dir+session_folder)
        for file in data_files:
            if "coding_section" in file:
                  coding_section_no = file.split("_")[2].split(".")[0]
                  df = pd.read_csv(data_dir+session_folder+"/"+file, sep="@")
                  df["coding_section_no"] = coding_section_no
                  dataframes.append(df)

# Concatenate all programming related-DataFrames into one large DataFrame
programming_full_data = pd.concat(dataframes, ignore_index=True)

In [ ]:
if debug:
    print(programming_full_data.columns)


SEMI-FINAL Step: Data sheets are mapped to treatment conditions


MAKE SURE TO ONLY DO THIS ONCE

In [ ]:
#There's a participant whose ID that screwed up so I fix it here
programming_full_data.loc[programming_full_data['participant_id'] == 'J210', 'participant_id'] = 'J21O'
nback_full_data.loc[nback_full_data['participant_id'] == 'J210', 'participant_id'] = 'J21O'

programming_full_data.loc[programming_full_data['participant_id'] == 327, 'participant_id'] = 'O327'
nback_full_data.loc[nback_full_data['participant_id'] == 327, 'participant_id'] = 'O327'

#There's a participant whose section number I fucked up (need to change from 3 to 2)
programming_full_data.loc[(programming_full_data['participant_id'] == 'P6XQ') & (programming_full_data['session'] == 3), 'session'] = 2
nback_full_data.loc[(nback_full_data['participant_id'] == 'P6XQ') & (nback_full_data['session'] == 3), 'session'] = 2

In [ ]:
# Load second CSV (Accuracy Data)
df_acc = pd.read_csv(program_grades)

# Melt accuracy data: Convert wide format to long format
df_acc_long = df_acc.melt(id_vars=["participant_id"], var_name="problem_id", value_name="accuracy")


# Ensure problem_id format matches in both DataFrames (if needed)
df_acc_long["problem_id"] = df_acc_long["problem_id"].astype(str)
#df_rt["problem_id"] = df_rt["problem_id"].astype(str)

# Merge on participant_id and problem_id
programming_full_data = programming_full_data.merge(df_acc_long, on=["participant_id", "problem_id"], how="left")


Merging result from demographics to full data

In [ ]:
#load demographic survey/information
df_demo = pd.read_csv(demographics)

#merge demographics to full data
programming_full_data = programming_full_data.merge(df_demo, on="participant_id", how="left")
nback_full_data = nback_full_data.merge(df_demo, on="participant_id", how="left")



In [ ]:
#load treatment mapping data
treatment_map_df = pd.read_csv(treatment_map)
metadata_long = treatment_map_df.melt(
    id_vars=["ID"], 
    value_vars=["Session #1 TMS", "Session #2 TMS"],
    var_name="Session_Type",
    value_name="TMS_Type"
)

# Extract session number from the column name
metadata_long["Session"] = metadata_long["Session_Type"].str.extract(r"(\d+)").astype(int)

# Select only the relevant columns
metadata_long = metadata_long[["ID", "Session", "TMS_Type"]]

# Merge metadata with task-level data
nback_full_data = nback_full_data.merge(
    metadata_long, 
    left_on=["participant_id", "session"], 
    right_on=["ID", "Session"],
    how="left"
)

# Merge metadata with task-level data
programming_full_data = programming_full_data.merge(
    metadata_long, 
    left_on=["participant_id", "session"], 
    right_on=["ID", "Session"],
    how="left"
)

# Drop unnecessary columns
nback_full_data = nback_full_data.drop(columns=["ID", "Session"])
programming_full_data = programming_full_data.drop(columns=["ID", "Session"])

Merger TMS sensations

In [ ]:
#load treatment mapping data
pain_map_df = pd.read_csv(pain_reports)
pain_map_df["participant_id"] = pain_map_df["participant_id"].str.strip()

# Melt the dataframe from wide to long
long_df = pain_map_df.melt(id_vars="participant_id", var_name="measure", value_name="value")

# Extract TMS_Type and Symptom from the column name
long_df[["TMS_Type", "symptom"]] = long_df["measure"].str.extract(r"(WM|control)[ _](.+)")

 # Standardize TMS_Type to 'WM' and 'CM'
long_df["TMS_Type"] = long_df["TMS_Type"].replace({"WM": "WM", "control": "CM"})

# Pivot back so each symptom is a column
symptoms_df = long_df.pivot_table(
    index=["participant_id", "TMS_Type"],
    columns="symptom",
    values="value",
    aggfunc="first"
).reset_index()

# Optional: clean up column names
symptoms_df.columns.name = None


# Merge symptoms into n-back and programming datasets
nback_full_data = nback_full_data.merge(
    symptoms_df,
    on=["participant_id", "TMS_Type"],
    how="left"
)

programming_full_data = programming_full_data.merge(
    symptoms_df,
    on=["participant_id", "TMS_Type"],
    how="left"
)


# Step 4: Optional — preview and/or save
print(programming_full_data.head())
# merged_df.to_csv("merged_output.csv", index=False)



FINAL Step: Output full data per tasks

In [ ]:
nback_output = output_dir + "/nback-full-data.csv"
programming_output = output_dir + "/programming-full-data.csv"

if output_data:
    if os.path.exists(nback_output): 
        print(f"'{nback_output}' already exists - Do you mean to replace it?")
    else: nback_full_data.to_csv(nback_output, sep=",", index=False)

if output_data:
    if os.path.exists(programming_output): 
        print(f"'{programming_output}' already exists - Do you mean to replace it?")
    else: programming_full_data.to_csv(programming_output, sep=",", index=False)
